# Split scanned page on the longest vertical dark line

Pipeline:
1. Find the longest near-vertical dark line (allowed to be slightly curved / not perfectly orthogonal, but mostly uninterrupted).
2. Whiten that line in the image (without bleeding into nearby letters).
3. Cut the image along the (curved) line. The two halves get padded with `#E01AD1` so each half is a rectangle whose missing area is the parallelogram-ish wedge between the curve and the straight vertical.
4. Stack the right half **below** the left half and save the result.

Tuned for two-column Fraktur scans (Glück 1694) with a thin printed divider rule between columns. Works on both clean reprint scans and brown-paper originals (set `gray_mode='blue'` for the latter).

In [ ]:
import numpy as np
from PIL import Image
import cv2
from pathlib import Path
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

In [ ]:
# ---------------------------------------------------------------
# Tunable parameters
# ---------------------------------------------------------------
CONFIG = {
    # Pixel must be at most this bright (0-255, single-channel) to count as 'dark'.
    # Raise for faded scans / darker paper.
    'dark_threshold': 140,

    # Channel used for darkness detection.
    #   'gray' = luminance (default, good for clean reprints)
    #   'blue' = blue channel (better contrast on yellow/brown paper)
    'gray_mode': 'gray',

    # Restrict the search for the line's column to the central band of the page,
    # given as fractions of width. Two-column scans usually have the gutter here.
    'search_band': (0.30, 0.70),

    # Width (px, on each side) used when scoring a column against its neighbors.
    # The score = (dark count in column) - (mean dark count in flanking columns).
    # This favours THIN spikes (a printed divider rule) over WIDE dark bands
    # (text columns), which would otherwise dominate by sheer dark-pixel count.
    'flank_width': 30,

    # When tracing the line row-by-row, how far horizontally we may jump
    # between consecutive rows (px). Bigger -> tolerates wavier lines, but
    # also more likely to grab edge of nearby text. 4-6 is a good range.
    'trace_max_jump': 4,

    # If a row has no dark pixel near the running x, we may skip it. Maximum
    # number of consecutive 'gap' rows tolerated before the line is considered
    # broken. For a 2400-px page on brown paper, faded sections of 100-200 rows are common.
    'max_gap_rows': 200,

    # The traced x must stay within this many pixels of the seed column.
    # If a row's nearest dark pixel is further than this, treat it as a gap.
    # This stops the trace from wandering into nearby text edges when the
    # divider truly ends or fades out.
    'max_drift_from_seed': 15,

    # After tracing, trim rows at each end of the detected run whose x is
    # far from the median x. Helps clean up a few wandering rows where the
    # divider had ended but the trace caught text edges within the drift
    # tolerance. Set to None to disable.
    'edge_trim_px': 8,

    # When we whiten the line we paint a small horizontal stripe around each
    # traced point. To avoid hitting nearby letters we shrink this stripe in
    # rows where dark text is detected just to the left/right.
    'whiten_half_width': 3,
    'letter_guard_distance': 14,  # px to look left/right for nearby ink

    # Padding color for the parallelogram wedges.
    'pad_color_hex': '#E01AD1',

    # If True, show debug plot inline. Batch mode overrides this to False but
    # can save overlay PNGs to disk via 'save_debug_dir'.
    'debug': True,
    'save_debug_dir': None,  # str path or None
}

def hex_to_rgb(h):
    h = h.lstrip('#')
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

PAD_RGB = hex_to_rgb(CONFIG['pad_color_hex'])
PAD_RGB

In [ ]:
# ---------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------
def to_single_channel(img_rgb, mode='gray'):
    """Return a HxW uint8 array. 'gray' = standard luminance; 'blue' = B channel,
    which gives much better contrast against yellow/brown paper."""
    if mode == 'blue':
        return img_rgb[:, :, 2].copy()
    elif mode == 'red':
        return img_rgb[:, :, 0].copy()
    elif mode == 'green':
        return img_rgb[:, :, 1].copy()
    else:
        return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)


In [ ]:
# ---------------------------------------------------------------
# Step 1: find the longest vertical-ish dark line
# ---------------------------------------------------------------
def find_vertical_line(img_rgb, cfg=CONFIG):
    """Return (xs, (y0, y1), seed_x):
      xs    : length-H int array; xs[y] >= 0 if the line was found at row y, else -1
      y0,y1 : inclusive bounds of the longest continuous (gap-bridged) detected run
      seed_x: the column the seed locked onto (useful for debugging)
    """
    chan = to_single_channel(img_rgb, cfg.get('gray_mode', 'gray'))
    H, W = chan.shape
    x_lo = int(cfg['search_band'][0] * W)
    x_hi = int(cfg['search_band'][1] * W)
    dark = chan <= cfg['dark_threshold']

    # ---- 1a. Seed: thin-spike score (column dark-count minus dark flanks) ----
    col_dark = dark[:, x_lo:x_hi].sum(axis=0).astype(np.float32)
    flank = cfg.get('flank_width', 30)
    n = len(col_dark)
    pad_arr = np.pad(col_dark, flank, mode='edge')
    cs = np.concatenate(([0.0], np.cumsum(pad_arr, dtype=np.float64)))
    left_mean  = (cs[flank:flank+n]           - cs[0:n])           / flank
    right_mean = (cs[2*flank+1:2*flank+1+n]   - cs[flank+1:flank+1+n]) / flank
    flank_mean = (left_mean + right_mean) / 2.0
    score = col_dark - flank_mean
    seed_x = x_lo + int(np.argmax(score))

    # ---- 1b. Trace from a seed row outward in both directions ----
    half = cfg['trace_max_jump']
    max_drift = cfg['max_drift_from_seed']
    max_gap = cfg['max_gap_rows']
    xs = np.full(H, -1, dtype=np.int32)

    rows_with = np.where(dark[:, max(0, seed_x-half):seed_x+half+1].any(axis=1))[0]
    if len(rows_with) == 0:
        return xs, (0, -1), seed_x
    seed_y = int(rows_with[len(rows_with) // 2])  # median row -> robust starting point

    def trace(start_y, direction):
        x = seed_x
        gap = 0
        y = start_y
        while 0 <= y < H:
            lo = max(0, x - half)
            hi = min(W, x + half + 1)
            row_slice = dark[y, lo:hi]
            if row_slice.any():
                idxs = np.where(row_slice)[0] + lo
                x_new = int(idxs[np.argmin(np.abs(idxs - x))])
                if abs(x_new - seed_x) > max_drift:
                    # Almost certainly ink from a neighbouring text column; skip.
                    gap += 1
                    if gap > max_gap:
                        break
                else:
                    xs[y] = x_new
                    x = x_new
                    gap = 0
            else:
                gap += 1
                if gap > max_gap:
                    break
            y += direction

    trace(seed_y, +1)
    trace(seed_y - 1, -1)
    xs[seed_y] = seed_x

    # ---- 1c. Bridge small gaps and find the longest run ----
    present = xs >= 0
    bridged = present.copy()
    last_true = -1
    for i in range(H):
        if present[i]:
            if last_true >= 0 and (i - last_true - 1) <= max_gap:
                bridged[last_true+1:i] = True
            last_true = i

    best = (0, -1)
    cur_start = None
    for i in range(H):
        if bridged[i] and cur_start is None:
            cur_start = i
        elif not bridged[i] and cur_start is not None:
            if i - cur_start > best[1] - best[0]:
                best = (cur_start, i - 1)
            cur_start = None
    if cur_start is not None and (H - cur_start) > best[1] - best[0]:
        best = (cur_start, H - 1)

    y0, y1 = best
    if y1 < y0:
        return xs, best, seed_x

    # ---- 1d. Interpolate over small gaps inside the run ----
    ys_known = [y for y in range(y0, y1+1) if xs[y] >= 0]
    if ys_known:
        xs_known = [xs[y] for y in ys_known]
        for y in range(y0, y1+1):
            if xs[y] < 0:
                xs[y] = int(np.interp(y, ys_known, xs_known))

    # ---- 1e. Edge cleanup: trim rows at run boundaries that wander far
    #         from the median x (typical when the divider has ended and
    #         the trace caught a few text-edge pixels within drift tol). ----
    trim = cfg.get('edge_trim_px')
    if trim is not None and y1 > y0:
        seg = xs[y0:y1+1]
        med_x = int(np.median(seg[seg >= 0]))
        new_y0 = y0
        while new_y0 <= y1 and abs(int(xs[new_y0]) - med_x) > trim:
            xs[new_y0] = -1
            new_y0 += 1
        new_y1 = y1
        while new_y1 >= new_y0 and abs(int(xs[new_y1]) - med_x) > trim:
            xs[new_y1] = -1
            new_y1 -= 1
        best = (new_y0, new_y1)

    return xs, best, seed_x


In [ ]:
# ---------------------------------------------------------------
# Step 2: whiten the line, avoiding nearby letters
# ---------------------------------------------------------------
def whiten_line(img_rgb, xs, y0, y1, cfg=CONFIG):
    """Return a copy of img with the line painted #FFFFFF, but skipping the paint
    in rows where dark ink (presumably text) sits very close on either side.
    """
    out = img_rgb.copy()
    H, W, _ = out.shape
    chan = to_single_channel(img_rgb, cfg.get('gray_mode', 'gray'))
    dark = chan <= cfg['dark_threshold']
    hw = cfg['whiten_half_width']
    guard = cfg['letter_guard_distance']

    for y in range(y0, y1+1):
        x = xs[y]
        if x < 0:
            continue
        left_lo = max(0, x - hw - guard)
        left_hi = max(0, x - hw)
        right_lo = min(W, x + hw + 1)
        right_hi = min(W, x + hw + 1 + guard)
        letter_left = dark[y, left_lo:left_hi].any() if left_hi > left_lo else False
        letter_right = dark[y, right_lo:right_hi].any() if right_hi > right_lo else False

        local_hw = 0 if (letter_left or letter_right) else hw
        lo = max(0, x - local_hw)
        hi = min(W, x + local_hw + 1)
        out[y, lo:hi] = (255, 255, 255)
    return out


In [ ]:
# ---------------------------------------------------------------
# Step 3: split into two rectangles padded with #E01AD1
# ---------------------------------------------------------------
def split_on_curve(img_rgb, xs, y0, y1, pad_rgb=PAD_RGB):
    """Cut the image along the curve given by `xs[y]` for y in [y0, y1].
    For rows outside [y0, y1] (where the line wasn't detected) fall back to
    the median x of the detected segment so the cut still goes top-to-bottom.
    """
    H, W, _ = img_rgb.shape
    detected = xs[y0:y1+1] if y1 >= y0 else np.array([], dtype=np.int32)
    detected = detected[detected >= 0]
    if len(detected) == 0:
        raise ValueError('No line detected')
    fallback_x = int(np.median(detected))

    full_xs = np.where(xs >= 0, xs, fallback_x)

    left_w = int(full_xs.max())
    right_origin = int(full_xs.min())
    right_w = W - right_origin

    left = np.full((H, left_w, 3), pad_rgb, dtype=np.uint8)
    right = np.full((H, right_w, 3), pad_rgb, dtype=np.uint8)

    for y in range(H):
        cut = int(full_xs[y])
        if cut > 0:
            left[y, :cut] = img_rgb[y, :cut]
        if cut < W:
            dst_start = max(0, cut - right_origin)
            right[y, dst_start:dst_start + (W - cut)] = img_rgb[y, cut:]
    return left, right


In [ ]:
# ---------------------------------------------------------------
# Step 4: stack right under left and pad widths to match
# ---------------------------------------------------------------
def stack_left_over_right(left, right, pad_rgb=PAD_RGB):
    target_w = max(left.shape[1], right.shape[1])

    def pad_right_to(img, w):
        if img.shape[1] == w:
            return img
        pad = np.full((img.shape[0], w - img.shape[1], 3), pad_rgb, dtype=np.uint8)
        return np.concatenate([img, pad], axis=1)

    left_p = pad_right_to(left, target_w)
    right_p = pad_right_to(right, target_w)
    return np.concatenate([left_p, right_p], axis=0)


In [ ]:
# ---------------------------------------------------------------
# Glue: process a single image
# ---------------------------------------------------------------
def make_overlay(img_rgb, xs, y0, y1):
    dbg = img_rgb.copy()
    H, W, _ = dbg.shape
    if y1 >= y0:
        for y in range(y0, y1+1):
            x = xs[y]
            if x >= 0:
                dbg[y, max(0, x-2):min(W, x+3)] = (255, 0, 0)
    return dbg

def process_image(in_path, out_path, cfg=CONFIG):
    img = np.array(Image.open(in_path).convert('RGB'))
    xs, (y0, y1), seed_x = find_vertical_line(img, cfg)
    if y1 < y0:
        raise RuntimeError(f'Could not find a vertical line in {in_path}')

    if cfg.get('debug'):
        plt.figure(figsize=(8, 10))
        plt.imshow(make_overlay(img, xs, y0, y1))
        plt.title(f'{Path(in_path).name}: rows {y0}-{y1}/{img.shape[0]}, seed x={seed_x}')
        plt.axis('off')
        plt.show()

    if cfg.get('save_debug_dir'):
        ddir = Path(cfg['save_debug_dir'])
        ddir.mkdir(parents=True, exist_ok=True)
        Image.fromarray(make_overlay(img, xs, y0, y1)).save(
            ddir / (Path(in_path).stem + '_overlay.png'))

    whitened = whiten_line(img, xs, y0, y1, cfg)
    left, right = split_on_curve(whitened, xs, y0, y1)
    stacked = stack_left_over_right(left, right)

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(stacked).save(out_path)
    return out_path


In [ ]:
# Single-image quick test:
# out = process_image('input_pages/gold_00016.jpg', 'output_pages/gold_00016_split.png')
# print('Saved:', out)

# Batch process a folder

In [ ]:
SOURCE_DIR = Path('input_pages')         # folder with the original scans
TARGET_DIR = Path('output_pages')        # where stacked results go
DEBUG_DIR  = Path('output_pages/_debug') # red-line overlays for spot-checking
EXTS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}

batch_cfg = dict(CONFIG)
batch_cfg['debug'] = False
batch_cfg['save_debug_dir'] = str(DEBUG_DIR)
# For brown-paper editions, switch to blue channel:
# batch_cfg['gray_mode'] = 'blue'
# batch_cfg['dark_threshold'] = 150

TARGET_DIR.mkdir(parents=True, exist_ok=True)

files = sorted(p for p in SOURCE_DIR.iterdir() if p.suffix.lower() in EXTS)
print(f'Found {len(files)} image(s) in {SOURCE_DIR}')

errors = []
for i, src in enumerate(files, 1):
    dst = TARGET_DIR / f'{src.stem}_split.png'
    try:
        process_image(src, dst, batch_cfg)
        print(f'[{i}/{len(files)}] OK  {src.name} -> {dst.name}')
    except Exception as e:
        errors.append((src, e))
        print(f'[{i}/{len(files)}] ERR {src.name}: {e}')

print(f'\nDone. {len(errors)} error(s). Overlays in {DEBUG_DIR}/')
for s, e in errors:
    print(' -', s.name, '->', e)